# Programmieraufgabe 7

Dieses mal wenden wir uns wieder der Zykloide aus Aufgabe 3 zu, und lösen die dazugehörige Differentialgleichung mit einem Mehrschrittverfahren.

Tragen Sie zunächst in der folgenen Zelle Ihren Namen ein:

In [ ]:
# Numerik gewöhnlicher Differentialgleichungen
# Sommersemester 2026
# Übungsblatt 10 - Programmieraufgabe 7
#
# [Nachname], [Vorname]
# [Vorname.Nachname@uni-a.de]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.linalg as la

Im Folgenden sind die Parameter für einige Mehrschrittverfahren bereits für Sie vorimplemeniert.

In [ ]:
# Welche der Verfahren sind explizit, welche implizit? Wie können Sie das feststellen?
#
#

In [ ]:
def Adams_Bashforth_1():
    alpha = np.array([-1.0, 1.0])
    beta  = np.array([ 1.0, 0.0])
    return alpha, beta

def Adams_Bashforth_2():
    alpha = np.array([0.0, -1.0, 1.0])
    beta  = np.array([-0.5, 1.5, 0.0])
    return alpha, beta

def Adams_Bashforth_3():
    alpha = np.array([0.0, 0.0, -1.0, 1.0])
    beta  = np.array([5/12, -16/12, 23/12, 0.0])
    return alpha, beta

def Adams_Bashforth_4():
    alpha = np.array([0.0, 0.0, 0.0, -1.0, 1.0])
    beta  = np.array([-9/24, 37/24, -59/24, 55/24, 0.0])
    return alpha, beta

def Adams_Bashforth_5():
    alpha = np.array([0.0, 0.0, 0.0, 0.0, -1.0, 1.0])
    beta  = np.array([251/720, -1274/720, 2616/720, -2774/720, 1901/720, 0.0])
    return alpha, beta

def Nystroem_2():
    alpha = np.array ([-1.0, 0.0, 1.0])
    beta  = np.array ([ 0.0, 2.0, 0.0])
    return alpha, beta

def Adams_Moulton_0():
    alpha = np.array ([-1.0, 1.0])
    beta  = np.array ([ 0.0, 1.0])
    return alpha, beta

def Adams_Moulton_1():
    alpha = np.array ([0.0, -1.0, 1.0])
    beta  = np.array ([0.0,  0.5, 0.5])
    return alpha, beta

def Adams_Moulton_2():
    alpha = np.array ([0.0,  0.0, -1.0, 1.0])
    beta  = np.array ([0.0, -1/12, 8/12, 5/12])
    return alpha, beta

def Adams_Moulton_3():
    alpha = np.array ([0.0, 0.0, 0.0, -1.0, 1.0])
    beta  = np.array ([0.0, 1/24, -5/24, 19/24, 9/24])
    return alpha, beta

def Adams_Moulton_4():
    alpha = np.array ([0.0, 0.0, 0.0, 0.0, -1.0, 1.0])
    beta  = np.array ([0.0, -19/720, 106/720, -264/720, 646/720, 251/720])
    return alpha, beta

Wir wollen im Folgenden ein lineares $k$-Schrittverfahren implementieren. Dieses hat im Allgemeinen folgende Form:
$$\sum_{j=0}^k\alpha_j u_{i+j} = h \sum_{j=0}^k\beta_j f(t_{i+j}, u_{i+j}).$$

Betrachten wir zunächst ein explizites Verfahren. In diesem Fall können wir die Gleichung oben nach dem nächsten Schritt $u_{i+k}$ auflösen, und erhalten
$$u_{i+k} = \left(h \sum_{j=0}^{k-1}\beta_jf(t_{i+j},u_{i+j}) - \sum_{j=0}^{k-1}\alpha_j u_{i+j}\right) / \alpha_k.$$

Welche Daten werden also, neben den Parametern $h$, $\alpha_j$ und $\beta_j$, für jeden Schritt benötigt?

In [ ]:
## Geben Sie hier die Daten an, die für ein explizites Verfahren in jedem Schritt benötigt werden. Diese sollten also zwischengespeichert werden.
#
#

Im impliziten Fall ist die allgemeine Form
$$\alpha_k u_{i+k} - h\beta_k f(t_{i+j},u_{i+k}) = \left(h \sum_{j=0}^{k-1}\beta_jf(t_{i+j},u_{i+j}) - \sum_{j=0}^{k-1}\alpha_j u_{i+j}\right)$$
und für eine lineare Differentialgleichung mit $f(t, y) = Ay + g(t)$ erhalten wir
$$\alpha_k u_{i+k} - h\beta_k Au_{i+k}  = \left(h \sum_{j=0}^{k-1}\beta_jf(t_{i+j},u_{i+j}) + \beta_kg(t_{i+k}) - \sum_{j=0}^{k-1}\alpha_j u_{i+j}\right).$$

Welche Matrix müssen Sie also in jedem Schritt eines implizten Verfahrens invertieren?

In [ ]:
## Geben Sie hier die Matrix an, die für ein implizites Verfahren invertiert werden muss:
#
#

Implementieren Sie jetzt ein allgemeines lineares Mehrschrittverfahren. Beachten Sie dabei:
- Die Funktion soll sowohl mit expliziten, als auch mit impliziten Verfahren funktionieren.
- Die ersten $k$-Schritte müssen dabei mit einem anderen Verfahren berechnet werden. Da wir später den Effekt dieses Initialisierungsverfahrens untersuchen wollen, übergeben wir diese Methode als Parameter `init`. Dies soll eine Funktion sein, die die Parameter `(f, t, u, h)` erhält, und den nächsten Wert `u(t + h)` approximiert.
- Für den impliziten Fall lohnt es sich, die Inverse der Matrix vorzuberechnen. Sie können dafür z.b. `L = la.lu_factor(A)` verwenden, das Sie dann mit `y = la.lu_solve(L, b)` für passende `A` und `b` aufrufen. Alternativ können Sie auch andere Funktionen, wie zum Beispiel `np.linalg.solve()` verwenden.

In [ ]:
def multistep(Af, g, t0, t_final, y0, N, alpha, beta, init):
    '''
        Lineares k-Schrittverfahren zum Lösen der linearen Differentialgleichung
            y'(t) = Af * y(t) + g(t)
    
        Parameter:
          Af:      homogener Teil der Differentialgleichung als Matrix
          g:       inhomogener Teil der Differentialgleichung als Funktion der Zeit
          t0:      Startzeitpunkt
          t_final: Endzeitpunkt
          y0:      Anfangswert
          N:       Anzahl Schritte
          alpha:   Parameter des Mehrschrittverfahrens
          beta:    Parameter des Mehrschrittverfahrens
          init:    Initialisierungsmethode
        Rückgabewert:
          us:      Berechneter Lösungspfad
    '''

    m = y0.shape[0]
    k = alpha.shape[0] - 1
    assert N >= k, "Es müssen mindestens k Schritte durchgeführt werden!"
    assert alpha.shape[0] == beta.shape[0], "alpha und beta müssen die selbe Länge haben!"

    us = np.zeros((m, N+1))
    fs = np.zeros((m, N+1))

    h = (t_final - t0) / N
    us[:,0] = y0
    fs[:,0] = Af @ y0 + g(t0)
    ts = np.linspace(t0, t_final, N+1)

    # Initialisierung
    f = lambda t, y: Af @ y + g(t)
    for j in range(1, k):
        us ???
        fs ???

    implicit = ???
    if implicit: # Vorberechnen der inversen Matrix
        L = ???

    for j in range(k, N+1):
        ???

    return us

In den nächsten beiden Zellen finden Sie bereits bekannte Funktionen: Zum einen ein allgemeines Runge-Kutte Einschrittverfahren zusammen mit mehreren Butcher-Tabellen, die wir als Initialisierung für das Mehrschrittverfahren verwenden werden. Ausserdem stellt die zweite Zelle das benötigte System für die Zykloide bereit, zusammen mit deren analytischer Lösung zur Fehlerberechnung.

In [ ]:
def RK(f, t, y, h, Butcher):
    '''
        Berechnet mit dem expliziten Runge-Kutta-Verfahren mit 
        gegebener  Butcher Tabelle den nächsten Punkt der 
        Differentialgleichung y'(t) = f(t, y(t))
        Parameter:
            f:        rechte Seite der Differentialgleichung
            t:        aktueller Zeitpunkt
            y:        aktueller Funktionswert
            h:        Schrittweite
            Butcher:  Butcher-Tabelle (A, b, c)
        Rückgabewert:
            u:        Approximation von y(t + h)
        Beachte: Es wäre noch effizienter, würde diese Methode auch f(t, y) zurückgeben,
                 was ja sowieso schon berechnet wurde und im Mehrschrittverfahren nochmals
                 benötigt wird. In dieser Übungsaufgabe haben wir darauf verzichtet, um
                 die Komplexität nicht noch weiter zu erhöhen, aber in einer echten
                 Implementierung wäre das auf jeden Fall sinnvoll.
    '''
    A, b, c = Butcher
    m = b.shape[0]
    n = y.shape[0]
    k = np.zeros((n,m))
    k[:,0] = f(t, y)
    for i in range(1, m):
        k[:,i] = f(t + h*c[i], y + h*(k[:,:i] @ A[i,:i]))
    u = y + h * k @ b
    return u

def RK_Euler():
    A = np.zeros((1, 1))
    b = np.ones(1)
    c = np.zeros(1)
    return A, b, c

def RK_Euler_mod():
    A = np.zeros((2,2))
    A[1,0] = 0.5
    b = np.array([0.0, 1.0])
    c = np.array([0.0, 0.5])
    return A, b, c

def RK_classic():
    A = np.zeros((4, 4))
    A[1,0] = 0.5
    A[2,1] = 0.5
    A[3,2] = 1
    b = np.array([1/6, 1/3, 1/3, 1/6])
    c = np.array([0, 0.5, 0.5, 1])
    return A, b, c

In [ ]:
def Af_Zykloide():
    return np.array([
        [ 0., 0., 1., 0.], 
        [ 0., 0., 0., 1.],
        [-1., 0., 0., 0.],
        [ 0.,-1., 0., 0.],
    ])

def g_Zykloide(t):
    return np.array([0.0, 0.0, t, 0.0])
    
def analytical_solution_Zykloide(t, r):
    tt = 0.5 * np.pi - t
    return np.array([
        t  + r * np.cos(tt),
        r * np.sin(tt),
        1.0 + r * np.sin(tt),
        -r * np.cos(tt),
    ])

Vervollständigen Sie die folgenden Zellen, in denen verschiedene Runge-Kutta Verfahren als Initialisierung getestet werden. Experimentieren Sie mit verschiedenen Mehrschrittverfahren `method` und beschreiben Sie ihre Beobachtungen.

In [ ]:
# Parameter
r       = 1.3
t0      = 0.0
t_final = t0 + 4.0*np.pi
y0      = np.array ([0.0, r, 1.0+r, 0.0])
Ns      = np.array([100 * 2**i for i in range(9)])

#  analytische Lösung berechnen
N_ref = Ns[-1]
h_ref = (t_final - t0) / N_ref
u_ref = np.zeros((y0.shape[0], N_ref+1))
u_ref[:,0] = y0
for i in range(1, N_ref+1):
    t = t0 + i*h_ref
    u_ref[:,i] = ???

init_Euler      = lambda f, t, y, h: RK(f, t, y, h, RK_Euler())
init_Euler_mod  = lambda f, t, y, h: RK(f, t, y, h, RK_Euler_mod())
init_RK_classic = lambda f, t, y, h: RK(f, t, y, h, RK_classic())

In [ ]:
method = Adams_Bashforth_4

errors = np.zeros((4, Ns.shape[0]))
alpha, beta = method()
for i, N in enumerate(Ns):
    h  = (t_final - t0) / N
    u1 = multistep(Af_Zykloide(), g_Zykloide, t0, t_final, y0, N, alpha, beta, init_Euler)
    u2 = multistep(Af_Zykloide(), g_Zykloide, t0, t_final, y0, N, alpha, beta, init_Euler_mod)
    u3 = multistep(Af_Zykloide(), g_Zykloide, t0, t_final, y0, N, alpha, beta, init_RK_classic)

    errors[0, i] = np.linalg.norm(u_ref[:,-1] - u1[:,-1])
    errors[1, i] = ???
    errors[2, i] = ???

    plt.subplot(3, 3, i+1)
    plt.title(f'N={int(N)}', loc='left')
    plt.plot(u_ref[0,:], u_ref[1,:], color='cyan', linewidth=4)
    plt.plot(u3[0,:], u3[1,:], color='black')
    plt.xlim([-0.2,     t_final - t0 + 0.2])
    plt.ylim([-0.2 - r, 0.2 + r])
plt.tight_layout()

plt.figure()
plt.grid()
plt.loglog(Ns, errors[0,:], '-o', markersize=10, markeredgewidth=3, markeredgecolor='black', color='red',   linewidth=3)
plt.loglog(Ns, errors[1,:], '-x', markersize=10, markeredgewidth=3, markeredgecolor='black', color='green', linewidth=3)
plt.loglog(Ns, errors[2,:], '-+', markersize=10, markeredgewidth=3, markeredgecolor='black', color='blue',  linewidth=3)
plt.legend(['exp. Euler', 'mod. Euler', 'klass. R-K'])
plt.xlabel('$N$')
plt.ylabel('$||y(t_N)-y_h(t_N)||$')
plt.show()

In [ ]:
### Beschreiben Sie hier kurz ihre Beobachtungen. Welchen Einfluss haben die Initialisierungsmethoden auf die Konvergenz? Ist dieser Einfluss unterschiedlich, je nach (Konsistenzordnung der) Verfahren?
#
#